# Paired validation benchmark — MMUbiPred-compatible vs UbiFusionNet

This notebook compares models on identical protein-grouped development splits across three fixed seeds. Every run is development-only: it neither reads the independent test set nor performs a full-training refit. Use a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Repository access

For a private repository, add a fine-grained read-only token to Colab Secrets as `GITHUB_TOKEN` and enable notebook access.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. Add GITHUB_TOKEN to Colab Secrets for a private repository.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
print('Project:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt

In [ ]:
import json
import platform
import sys
import sklearn
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Select a GPU runtime before starting the benchmark.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Run the paired three-seed benchmark

This runs six development-only trainings: two models multiplied by three seeds. The runner saves a partial summary after every completed run.

In [ ]:
SUITE_CONFIG = 'experiment/configs/paired_baseline_v1.json'
run_live([
    sys.executable, '-u', 'experiment/scripts/run_validation_benchmark.py',
    '--suite', SUITE_CONFIG,
])

In [ ]:
SUMMARY_PATH = REPO_ROOT / 'experiment/outputs/benchmarks/paired_baseline_v1/summary.json'
summary = json.loads(SUMMARY_PATH.read_text())
display(summary['aggregates'])
display(summary['paired_differences'])

## Preserve compact benchmark evidence in Drive

In [ ]:
import shutil
from datetime import date

BENCHMARK_ROOT = SUMMARY_PATH.parent
DRIVE_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/benchmarks') / f'{date.today().isoformat()}-paired-baseline-v1'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for filename in ('summary.json', 'runs.tsv'):
    shutil.copy2(BENCHMARK_ROOT / filename, DRIVE_ROOT / filename)
for manifest in BENCHMARK_ROOT.glob('*/seed_*/run_manifest.json'):
    relative = manifest.relative_to(BENCHMARK_ROOT)
    destination = DRIVE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(manifest, destination)
for metrics in BENCHMARK_ROOT.glob('*/seed_*/validation_metrics.json'):
    relative = metrics.relative_to(BENCHMARK_ROOT)
    destination = DRIVE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metrics, destination)
print('Saved compact benchmark evidence to:', DRIVE_ROOT)